## Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect 
signatures of anomalous or strongly anomalous diffusion. The analysis 
is performed first on the full aggregated dataset, then compared across 
distance groups. For each signal, the $q$-th order moment is computed 
at different time scales $\tau$:

$$M_q(\tau) = \langle |a|^q \rangle_\tau$$

If the process exhibits scaling, the moments obey a power law:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Deviations from 
linearity indicate anomalous or strongly anomalous diffusion.

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src.signals_scaling import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         trim_to_event_window, compute_increment_tail_exponents)
from src.plot_settings import set_plot_style
from src.plots import plot_onset_diagnostic, plot_onset_distribution, plot_increment_distributions, plot_r2_diagnostic
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

### Computation of q-th order moments

Moments are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

In [ ]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Compute moments — global signal
df_moments_agg = compute_moment_scaling(df_global, q_values, tau_values)
print("Shape:", df_moments_agg.shape)

try:
    df_moments_agg.to_parquet('../data/processed/moments_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

A plot of $\zeta(q)$ vs $q$ is produced for each signal, with the reference 
line $\zeta(q) = q/2$ corresponding to normal diffusion.

In [ ]:
# Compute scaling exponents on full aggregated dataset
df_exponents_agg = compute_scaling_exponents(df_moments_agg,
                                              output_dir=FIGURES_DIR / 'scaling')
try:
    df_exponents_agg.to_parquet('../data/processed/scaling_exponents_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing a linear 
and a quadratic fit using AIC. A piecewise linear fit is also performed 
to detect the presence of two distinct scaling regimes:

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes.

In [ ]:
# Linearity check on full aggregated dataset
df_linearity_agg = test_scaling_linearity(df_exponents_agg,
                                           output_dir=FIGURES_DIR / 'scaling')
try:
    df_linearity_agg.to_parquet('../data/processed/scaling_linearity_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

# Piecewise linear fit on full aggregated dataset
df_piecewise_agg = fit_piecewise_scaling(df_exponents_agg,
                                          output_dir=FIGURES_DIR / 'scaling')
try:
    df_piecewise_agg.to_parquet('../data/processed/scaling_piecewise_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")